# 00. Data EDA and Schema Walkthrough

This notebook is the entry point for a full reproduction walk-through.
It does not assume that you already know the processed tables. We inspect
the schemas, row counts, date ranges, and missingness before computing
any PEC metrics.

**Goal:** understand what each processed table contributes to the paper.


<!-- PEC_RERUN_SAFETY_V1 -->

## Execution safety note

This notebook is an authorized private rerun notebook. It can use real processed data when `PEC_GAP_PROCESSED_DIR` or `PEC_GAP_PRIVATE_ROOT` is set, but notebook-generated outputs should stay under `tmp/repro_deep/...` unless you intentionally publish approved aggregate artifacts. The manuscript-facing `reports/tables/`, `reports/figures/`, and Overleaf folders are treated as read-only reference artifacts in this notebook.


<!-- PEC_NOTEBOOK_EXPLANATION_V1:eda_guide -->

## Reader guide: what to verify before analysis

This notebook is the audit's **data health checkpoint**. Before computing any PEC metric, we need to know whether the processed tables cover the intended study window and contain the fields needed to link users, articles, recommendations, clicks, surfaces, and article metadata.

Read this notebook with three questions in mind:

1. Do the tables cover the expected January-June 2026 observation window?
2. Are the key identifiers available for joining traces across layers?
3. Are there missingness or type issues that would limit a claim later in the paper?

This is not just administrative EDA. In an audit paper, data availability defines what can and cannot be claimed.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:eda_layers -->

## PEC layer mapping used in the EDA

- **Profiles:** stated category/source selections and weighted profile-state fields. These become the preference layer $P$.
- **Recommendation items / rows:** user-date ranked lists. These become the logged exposure layer $E$.
- **Clicks / behavior:** observed click consumption and surface metadata. These become the consumption layer $C$ and surface-pathway layer $S$.
- **News metadata:** article category, publisher, language, publication time, and related item attributes. These labels make the P/E/C layers comparable.

If a layer lacks identifiers or metadata, later alignment and traceability metrics become weaker or impossible.

In [ ]:
from pathlib import Path
import os
import ast
import json
import math
import sys
import warnings
from collections import Counter


def find_repo_root() -> Path:
    """Find the public/release repository root, independent of private data location."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "reports").exists() and (candidate / "notebooks").exists() and (candidate / "scripts").exists():
            return candidate
    env_repo = os.environ.get("PEC_GAP_REPO_ROOT")
    if env_repo and Path(env_repo).exists():
        return Path(env_repo)
    raise FileNotFoundError("Could not find the PEC repository root. Set PEC_GAP_REPO_ROOT if needed.")


def find_processed_dir(repo_root: Path) -> Path:
    """Find authorized private processed tables without requiring them to live in the repo."""
    exact = os.environ.get("PEC_GAP_PROCESSED_DIR")
    if exact:
        return Path(exact)
    private_root = os.environ.get("PEC_GAP_PRIVATE_ROOT")
    if private_root:
        return Path(private_root) / "data" / "processed"
    local_config = repo_root / ".pecgap_local_paths.json"
    if local_config.exists():
        try:
            cfg = json.loads(local_config.read_text(encoding="utf-8"))
            if cfg.get("processed_dir"):
                return Path(cfg["processed_dir"])
            if cfg.get("private_root"):
                return Path(cfg["private_root"]) / "data" / "processed"
        except Exception as exc:
            print(f"Warning: could not read {local_config}: {exc}")
    return repo_root / "data" / "processed"


ROOT = find_repo_root()
PROCESSED = find_processed_dir(ROOT)
PKG = ROOT / ".python_packages"
if PKG.exists() and sys.version_info[:2] == (3, 12) and str(PKG) not in sys.path:
    # The project package cache is compiled for Python 3.12.
    # Anaconda/Jupyter kernels on this machine are often Python 3.10, so
    # we only add the cache when the ABI matches.
    sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Read-only manuscript/reference artifacts. These are used for comparison.
# For authorized reruns, prefer the private project's reports/tables when the
# processed data live outside this public/release repo. This keeps row-level
# validation tables out of GitHub while still letting the notebooks compare
# regenerated values against the private run artifacts.
PRIVATE_ROOT = PROCESSED.parent.parent if PROCESSED.name == "processed" and PROCESSED.parent.name == "data" else ROOT
PRIVATE_TABLES = PRIVATE_ROOT / "reports" / "tables"
PRIVATE_FIGURES = PRIVATE_ROOT / "reports" / "figures"
PUBLIC_TABLES = ROOT / "reports" / "tables"
PUBLIC_FIGURES = ROOT / "reports" / "figures"
TABLES = PRIVATE_TABLES if PRIVATE_TABLES.exists() else PUBLIC_TABLES
FIGURES = PRIVATE_FIGURES if PRIVATE_FIGURES.exists() else PUBLIC_FIGURES

# Notebook-generated outputs go here by default, not into reports/tables or Overleaf.
NOTEBOOK_STEM = "00_data_eda_and_schema"
SCRATCH = ROOT / "tmp" / "repro_deep" / NOTEBOOK_STEM
SCRATCH_TABLES = SCRATCH / "tables"
SCRATCH_FIGURES = SCRATCH / "figures"
SCRATCH_TABLES.mkdir(parents=True, exist_ok=True)
SCRATCH_FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Processed data: {PROCESSED}")
print(f"Processed data exists: {PROCESSED.exists()}")
print(f"Read-only reference tables: {TABLES}")
print(f"Scratch output tables: {SCRATCH_TABLES}")
print(f"Scratch output figures: {SCRATCH_FIGURES}")
if not PROCESSED.exists():
    display(Markdown(
        "**Private processed data not found.** This notebook is real rerun code, "
        "but it needs authorized processed CSVs. Set `PEC_GAP_PROCESSED_DIR` to the exact "
        "processed-data folder, or set `PEC_GAP_PRIVATE_ROOT` to a root containing `data/processed/`."
    ))


In [ ]:
files = {
    "profiles": "user_profiles_clean.csv",
    "clicks": "click_events_enriched.csv",
    "recommendation_items": "recommendation_items_enriched.csv",
    "recommendation_rows": "recommendation_rows_mapped.csv",
    "news": "news_master.csv",
    "behavior": "behavior_events_dedup.csv",
}

data = {}
for name, filename in files.items():
    path = PROCESSED / filename
    data[name] = pd.read_csv(path, dtype=str).fillna("")

shapes = pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": df.shape[1]} for name, df in data.items()]
).sort_values("table")
display(shapes)


<!-- PEC_NOTEBOOK_EXPLANATION_V2:dataset_inventory -->

## 먼저 데이터셋을 “분석 재료”로 이해하기

여기서 중요한 것은 단순히 행 수를 보는 것이 아니라, 각 CSV가 PEC 논문의 어느 관측 층을 담당하는지 이해하는 것이다.

- `user_profiles_clean.csv`는 사용자별 **현재 프로필 상태**다. 여기서 stated categories, preferred sources, weighted profile state가 나온다.
- `behavior_events_dedup.csv`는 앱 사용 로그 전체에 가까운 **행동 이벤트 로그**다. 모든 이벤트가 분석에 직접 들어가지는 않지만, 클릭 표면과 사용 패턴의 출발점이다.
- `click_events_enriched.csv`는 행동 로그에서 클릭 이벤트를 deduplicate하고 기사 메타데이터를 붙인 **소비 layer**다.
- `recommendation_rows_mapped.csv`는 사용자-날짜별 추천 리스트 원본 행이다. 한 행 안에 여러 `news_id`가 들어 있다.
- `recommendation_items_enriched.csv`는 추천 리스트를 item-level로 펼친 **logged exposure layer**다. `rank`와 `recommendation_date`가 핵심이다.
- `news_master.csv`는 기사 category, publisher, publication date 같은 공통 라벨을 제공한다.

즉, 원천에서 바로 Jaccard나 entropy가 나오는 것이 아니라, 이 테이블들을 사용자(`user_id`), 기사(`news_id`), 날짜(`event_date_iso`/`recommendation_date`) 기준으로 조인하고 집계해서 RQ별 분석 테이블을 만든다.

In [ ]:
# Column inventory and human-readable data dictionary.
# 이 셀은 “어떤 CSV에 어떤 컬럼이 있고, 논문 분석에서 어떤 역할을 하는지”를 먼저 지도처럼 보여준다.
# 모든 컬럼을 다 외울 필요는 없지만, join key와 paper metric에 쓰이는 핵심 컬럼은 반드시 이해해야 한다.

dataset_guide = pd.DataFrame([
    {
        "table": "profiles",
        "file": "user_profiles_clean.csv",
        "grain": "one row per user/profile snapshot",
        "PEC role": "Preference layer P",
        "core keys": "user_id, profile_id/id",
        "what it contributes": "stated category/source sets, weighted profile-state distributions, onboarding/profile metadata",
        "main downstream use": "RQ1 set overlap; RQ1 weighted cosine; RQ4 profile-state features",
    },
    {
        "table": "behavior",
        "file": "behavior_events_dedup.csv",
        "grain": "one row per app/GA4 behavior event",
        "PEC role": "raw behavior and surface context source",
        "core keys": "user_id, event_timestamp/event_date_iso, news_id when present",
        "what it contributes": "event_name, screen/source/section fields, device/session/use metadata",
        "main downstream use": "source material for click extraction, surface assignment, usage summaries",
    },
    {
        "table": "clicks",
        "file": "click_events_enriched.csv",
        "grain": "one row per deduplicated/enriched clicked article event",
        "PEC role": "Consumption layer C plus surface pathway S",
        "core keys": "user_id, news_id, event_date_iso",
        "what it contributes": "clicked articles, clicked categories/publishers, click-time surface metadata",
        "main downstream use": "RQ1 consumed category sets; RQ2 clicked article matching; RQ3 click distributions; RQ4 behavior features",
    },
    {
        "table": "recommendation_rows",
        "file": "recommendation_rows_mapped.csv",
        "grain": "one row per user-date recommendation list record",
        "PEC role": "logged recommendation list container",
        "core keys": "id, user_id, recommendation_date_iso",
        "what it contributes": "the original list-level record and ordered news_ids string",
        "main downstream use": "audit trail for how item-level recommendation exposure was expanded",
    },
    {
        "table": "recommendation_items",
        "file": "recommendation_items_enriched.csv",
        "grain": "one row per recommended article item within a list",
        "PEC role": "Logged exposure layer E",
        "core keys": "recommendation_row_id, user_id, recommendation_date, rank, news_id",
        "what it contributes": "ranked exposure candidates with category/publisher labels",
        "main downstream use": "RQ2 same-day hit/top-k; RQ3 exposure distributions; RQ4 exposure features",
    },
    {
        "table": "news",
        "file": "news_master.csv",
        "grain": "one row per news article",
        "PEC role": "metadata bridge across P/E/C",
        "core keys": "news_id",
        "what it contributes": "article title, category, publisher_id, publication time, text length/language",
        "main downstream use": "labels clicked/recommended articles so distributions are comparable",
    },
])

display(Markdown("### Dataset guide: what each processed table means"))
display(dataset_guide)

inventory = []
for name, df in data.items():
    inventory.append({
        "table": name,
        "n_columns": df.shape[1],
        "first_columns": ", ".join(df.columns[:25]) + (" ..." if df.shape[1] > 25 else ""),
    })
inventory = pd.DataFrame(inventory).sort_values("table")
display(Markdown("### Column inventory: quick view of available fields"))
display(inventory)

column_notes = [
    ("profiles", "user_id", "사용자 단위 join key", "click/recommendation tables와 사용자 기준으로 연결한다."),
    ("profiles", "interested_categories_user_selection", "명시적 stated category set", "RQ1 top-category match/Jaccard의 stated preference set 구성에 우선 사용한다."),
    ("profiles", "preferred_news_sources_user_selection", "명시적 preferred source set", "publisher/source overlap sensitivity에 사용한다."),
    ("profiles", "interested_categories", "weighted category profile state", "category preference-click cosine과 RQ4 weighted-profile features에 사용한다."),
    ("profiles", "preferred_news_sources", "weighted source profile state", "publisher/source cosine과 source-space 비교에 사용한다."),
    ("behavior", "event_name", "행동 이벤트 종류", "news_click 같은 클릭 이벤트와 일반 앱 이벤트를 구분한다."),
    ("behavior", "event_dt_kst / event_date_iso", "행동 발생 시각/날짜", "분석 기간과 same-day matching의 날짜 기준을 만든다."),
    ("behavior", "p_source / section / is_headline / position", "click surface metadata", "headline/category/home-feed/newsroom/article-detail 등 surface assignment의 원재료다."),
    ("clicks", "news_id", "클릭된 기사 ID", "news_master 및 recommendation_items와 기사 기준으로 연결한다."),
    ("clicks", "category_final", "클릭 기사 category", "RQ1 clicked category set, RQ3 click distribution, RQ4 label/features에 사용한다."),
    ("clicks", "publisher_id_final", "클릭 기사 publisher/source", "publisher/source overlap 및 distribution 계산에 사용한다."),
    ("clicks", "news_matched", "기사 메타데이터 연결 성공 여부", "분석 가능한 clicked article만 필터링할 때 사용한다."),
    ("recommendation_rows", "news_ids", "한 추천 리스트 안의 기사 ID 문자열", "item-level recommendation_items로 explode되기 전 원본 형태다."),
    ("recommendation_items", "recommendation_date", "추천 리스트 날짜", "click event date와 same-day matching한다."),
    ("recommendation_items", "rank", "추천 리스트 내 순위", "top-k hit/lift 분석에 사용한다."),
    ("recommendation_items", "news_id", "추천된 기사 ID", "clicks.news_id와 비교해 clicked article이 추천 로그에 있었는지 확인한다."),
    ("recommendation_items", "category / publisher_id", "추천 기사 category/source", "RQ3 exposure distribution과 RQ4 exposure features에 사용한다."),
    ("news", "news_id", "기사 메타데이터 join key", "click/recommendation item에 category와 publisher를 붙이는 기준이다."),
    ("news", "category / publisher_id", "공통 article labels", "P/E/C 분포를 같은 label space에서 비교하게 해준다."),
]
column_dictionary = pd.DataFrame(column_notes, columns=["table", "column", "meaning", "paper use"])
display(Markdown("### Core column dictionary: the columns you should recognize"))
display(column_dictionary)


In [ ]:
validation_path = PROCESSED / "validation_summary.json"
if validation_path.exists():
    validation = json.loads(validation_path.read_text())
    display(pd.DataFrame([validation]).T.rename(columns={0: "value"}))
else:
    print("No validation_summary.json found.")


In [ ]:
def date_range(df, col):
    parsed = pd.to_datetime(df[col], errors="coerce")
    return {
        "date_col": col,
        "min": parsed.min(),
        "max": parsed.max(),
        "valid_rows": int(parsed.notna().sum()),
    }

date_checks = []
candidates = {
    "news": ["published_at", "dt", "created_month"],
    "clicks": ["event_date_iso", "event_month", "event_dt_kst"],
    "recommendation_items": ["recommendation_date"],
    "behavior": ["event_date_iso", "event_month", "event_dt_kst"],
    "profiles": ["created_at", "updated_at", "onboarding_completed_at"],
}
for table, cols in candidates.items():
    for col in cols:
        if col in data[table].columns:
            row = date_range(data[table], col)
            row["table"] = table
            date_checks.append(row)
display(pd.DataFrame(date_checks)[["table", "date_col", "min", "max", "valid_rows"]])


In [ ]:
# Missingness for the fields that drive the paper.
key_cols = {
    "profiles": [
        "user_id", "interested_categories", "preferred_news_sources",
        "interested_categories_user_selection", "preferred_news_sources_user_selection",
        "tier", "platform", "preferred_language",
    ],
    "clicks": [
        "user_id", "event_date_iso", "news_id", "news_matched",
        "category_final", "publisher_id_final", "message_type", "p_source",
        "section", "is_headline", "position",
    ],
    "recommendation_items": [
        "user_id", "recommendation_date", "rank", "news_id", "news_matched",
        "category", "publisher_id",
    ],
    "news": ["news_id", "publisher_id", "category", "published_at", "article_text_len"],
    "behavior": ["user_id", "event_name", "event_date_iso", "news_id", "section", "p_source"],
}
rows = []
for table, cols in key_cols.items():
    df = data[table]
    for col in cols:
        if col in df.columns:
            blank = df[col].astype(str).str.strip().eq("")
            rows.append({
                "table": table,
                "column": col,
                "blank_rows": int(blank.sum()),
                "blank_pct": float(blank.mean() * 100),
                "unique_values": int(df[col].nunique(dropna=False)),
            })
display(pd.DataFrame(rows).sort_values(["table", "blank_pct"], ascending=[True, False]))


In [ ]:
def safe_parse(value):
    '''Parse JSON-like strings stored in CSV cells.''';
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip()
    if not text:
        return None
    for parser in (json.loads, ast.literal_eval):
        try:
            return parser(text)
        except Exception:
            continue
    return None


def truthy(series: pd.Series) -> pd.Series:
    '''Normalize common true-like CSV values to booleans.''';
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def pref_set(row: pd.Series, primary: str, fallback: str) -> set[str]:
    '''Return a stated preference set from explicit selections or weighted profile state.

    primary is usually the user-selection column. fallback is the current weighted
    profile-state column. For a weighted dictionary, small positive weights are
    treated as selected dimensions for set-overlap analyses.
    '''
    parsed = safe_parse(row.get(primary, ""))
    if parsed is None:
        parsed = safe_parse(row.get(fallback, ""))
    if isinstance(parsed, dict):
        out = set()
        for key, value in parsed.items():
            try:
                if float(value) > 0.01:
                    out.add(str(key))
            except Exception:
                out.add(str(key))
        return out or set(map(str, parsed.keys()))
    if isinstance(parsed, list):
        out = set()
        for item in parsed:
            if isinstance(item, dict) and "value" in item:
                out.add(str(item["value"]))
            else:
                out.add(str(item))
        return out
    return set()


def normalize_dist(values: dict[str, float]) -> dict[str, float]:
    clean = {}
    for key, value in values.items():
        try:
            val = float(value)
        except Exception:
            continue
        if key and val > 0:
            clean[str(key)] = val
    total = sum(clean.values())
    if total <= 0:
        return {}
    return {key: val / total for key, val in clean.items()}


def parse_weight_dist(value) -> dict[str, float]:
    '''Parse a weighted preference dictionary such as {'Politics': 0.7, ...}.''';
    parsed = safe_parse(value)
    if isinstance(parsed, dict):
        return normalize_dist({str(k): v for k, v in parsed.items()})
    if isinstance(parsed, list):
        vals = []
        for item in parsed:
            if isinstance(item, dict) and "value" in item:
                vals.append(str(item["value"]))
            else:
                vals.append(str(item))
        if vals:
            return {key: 1 / len(vals) for key in vals if key}
    return {}


def norm_entropy(values) -> float:
    '''Normalized entropy in [0, 1]. Higher means more diverse.''';
    values = pd.Series(values).dropna().astype(str)
    values = values[values != ""]
    if values.empty:
        return np.nan
    counts = values.value_counts().to_numpy(dtype=float)
    probs = counts / counts.sum()
    ent = -(probs * np.log(probs)).sum()
    max_ent = np.log(len(counts)) if len(counts) > 1 else 1.0
    return float(ent / max_ent) if max_ent else 0.0


def norm_entropy_dist(dist: dict[str, float]) -> float:
    values = np.array([v for v in dist.values() if v > 0], dtype=float)
    if len(values) == 0:
        return 0.0
    values = values / values.sum()
    ent = float(-(values * np.log(values)).sum())
    denom = math.log(len(values)) if len(values) > 1 else 1.0
    return ent / denom if denom else 0.0


def top_share(values) -> float:
    '''Share of the most common item. Higher means more concentrated.''';
    values = pd.Series(values).dropna().astype(str)
    values = values[values != ""]
    if values.empty:
        return np.nan
    return float(values.value_counts().iloc[0] / len(values))


def hhi(values) -> float:
    '''Herfindahl-Hirschman concentration index. Higher means more concentrated.''';
    values = pd.Series(values).dropna().astype(str)
    values = values[values != ""]
    if values.empty:
        return np.nan
    probs = values.value_counts(normalize=True).to_numpy(dtype=float)
    return float(np.sum(probs * probs))


def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return np.nan
    return len(a & b) / len(a | b)


def distribution_from_values(values) -> dict[str, float]:
    series = pd.Series(values).dropna().astype(str)
    series = series[series != ""]
    if series.empty:
        return {}
    counts = series.value_counts()
    return {str(k): float(v) / float(counts.sum()) for k, v in counts.items()}


def cosine(a: dict[str, float], b: dict[str, float]) -> float:
    keys = sorted(set(a) | set(b))
    if not keys:
        return np.nan
    av = np.array([a.get(k, 0.0) for k in keys], dtype=float)
    bv = np.array([b.get(k, 0.0) for k in keys], dtype=float)
    denom = np.linalg.norm(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom > 0 else np.nan


def js_divergence(a: dict[str, float], b: dict[str, float]) -> float:
    '''Jensen-Shannon divergence between two discrete distributions.''';
    keys = sorted(set(a) | set(b))
    if not keys:
        return np.nan
    p = np.array([a.get(k, 0.0) for k in keys], dtype=float)
    q = np.array([b.get(k, 0.0) for k in keys], dtype=float)
    if p.sum() <= 0 or q.sum() <= 0:
        return np.nan
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    def kl(x, y):
        mask = x > 0
        return float((x[mask] * np.log2(x[mask] / y[mask])).sum())
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def dist_top_key(dist: dict[str, float]) -> str:
    return max(dist.items(), key=lambda item: item[1])[0] if dist else ""


def classify_surface(row: pd.Series) -> str:
    '''Map raw app parameters to the surface categories used in the paper.''';
    source = str(row.get("p_source", "") or "").lower()
    section = str(row.get("section", "") or "").lower()
    message = str(row.get("message_type", "") or "").lower()
    is_headline = str(row.get("is_headline", "") or "").lower() in {"1.0", "1", "true", "yes"}
    if "push" in message or "notification" in source or "notification" in section:
        return "notification"
    if is_headline or "headline" in section:
        return "headline"
    if section.startswith("category_"):
        return "category"
    if "for_you" in section or source == "home":
        return "home/feed"
    if "related" in section or source == "news_detail":
        return "article-detail"
    if "publisher" in section or source in {"newsroom", "source"}:
        return "newsroom"
    if "podcast" in section or "podcast" in source:
        return "podcast"
    if "insight" in section or "insight" in source:
        return "insights"
    if "search" in section or source == "search":
        return "search"
    return "other/unknown"


def mask_id(value: str) -> str:
    '''Mask user identifiers before showing examples in a notebook.''';
    text = str(value)
    if len(text) <= 8:
        return text[:2] + "***"
    return text[:4] + "***" + text[-4:]


In [ ]:
# Core observation counts used by the paper.
clicks = data["clicks"]
recs = data["recommendation_items"]
profiles = data["profiles"]
news = data["news"]

core_counts = {
    "profiles": len(profiles),
    "profiles_with_user_id": profiles["user_id"].astype(str).str.len().gt(0).sum(),
    "matched_click_rows": int(truthy(clicks["news_matched"]).sum()),
    "users_with_matched_clicks": clicks.loc[truthy(clicks["news_matched"]), "user_id"].nunique(),
    "matched_recommendation_items": int(truthy(recs["news_matched"]).sum()),
    "users_with_matched_recommendations": recs.loc[truthy(recs["news_matched"]), "user_id"].nunique(),
    "news_records": len(news),
    "news_with_article_text": news.get("article_text_len", pd.Series("", index=news.index)).astype(str).str.strip().ne("").sum(),
}
display(pd.DataFrame([core_counts]).T.rename(columns={0: "count"}))


In [ ]:
# Monthly volume check. This helps verify that the Jan-Jun 2026 news/content
# window and behavior logs line up before deeper analysis.
monthly = []
for table, df, col in [
    ("clicks", clicks, "event_month"),
    ("recommendation_items", recs, "recommendation_date"),
    ("news", news, "created_month"),
    ("behavior", data["behavior"], "event_month"),
]:
    if col not in df.columns:
        continue
    if table == "recommendation_items":
        month = pd.to_datetime(df[col], errors="coerce").dt.strftime("%Y-%m")
    else:
        month = df[col].astype(str).str.slice(0, 7)
    tmp = month.value_counts().sort_index().rename_axis("month").reset_index(name="rows")
    tmp["table"] = table
    monthly.append(tmp)
monthly = pd.concat(monthly, ignore_index=True)
display(monthly.pivot_table(index="month", columns="table", values="rows", aggfunc="sum").fillna(0).astype(int))

ax = monthly.pivot_table(index="month", columns="table", values="rows", aggfunc="sum").fillna(0).plot(
    kind="bar", figsize=(12, 4), width=0.82
)
ax.set_title("Monthly rows by processed table")
ax.set_xlabel("Month")
ax.set_ylabel("Rows")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


**What you should take away from Notebook 00**

- The paper uses processed tables, not raw Cloud SQL exports.
- The audit links user profiles, logged recommendation items, click events, surface metadata, and news metadata.
- `interested_categories` / `preferred_news_sources` are weighted profile-state fields.
- `*_user_selection` columns preserve the explicit user-selected preference sets.
- Later notebooks deliberately keep these representations separate.


## Masked row samples: what each table looks like

아래 샘플은 원자료를 그대로 공개하려는 것이 아니라, 각 테이블의 “행 단위 의미”를 이해하기 위한 마스킹된 예시다.

- 개인 식별 가능성이 있는 `email`, `password_hash`, 이름, 이미지 URL 등은 가린다.
- 긴 본문/JSON은 앞부분만 잘라서 보여준다.
- ID는 앞 몇 글자만 남긴다.

샘플을 볼 때 핵심 질문은 “이 행 하나가 무엇을 의미하는가?”이다. 예를 들어 `clicks` 한 행은 한 사용자가 한 날짜에 한 기사를 클릭한 소비 이벤트이고, `recommendation_items` 한 행은 한 사용자의 특정 날짜 추천 리스트 안에 들어간 한 기사 item이다.

In [ ]:
# Masked sample rows for learning the table grain.
# 이 셀은 샘플을 보여주고, 같은 내용을 tmp/repro_deep/.../sample_<table>_masked.csv로 저장한다.

def mask_value(col, value):
    text = "" if pd.isna(value) else str(value)
    low = col.lower()
    if any(token in low for token in ["email", "password", "hash", "token", "image", "name"]):
        return "[masked]"
    if low in {"user_id", "id", "profile_id", "recommendation_row_id", "news_id", "publisher_id", "publisher_id_final", "publisher_id_news", "publisher_id_behavior"} or low.endswith("_id"):
        return text[:8] + "..." if text else ""
    if len(text) > 120:
        return text[:117] + "..."
    return text

sample_columns = {
    "profiles": ["user_id", "tier", "interested_categories_user_selection", "preferred_news_sources_user_selection", "interested_categories", "preferred_language", "platform"],
    "behavior": ["event_date_iso", "event_dt_kst", "user_id", "event_name", "news_id", "p_source", "section", "is_headline", "position", "device_category"],
    "clicks": ["event_date_iso", "event_dt_kst", "user_id", "event_name", "news_id", "category_final", "publisher_id_final", "p_source", "section", "is_headline", "position", "news_matched"],
    "recommendation_rows": ["id", "user_id", "recommendation_date_iso", "news_ids", "profile_id"],
    "recommendation_items": ["recommendation_row_id", "user_id", "recommendation_date", "rank", "news_id", "category", "publisher_id", "news_matched"],
    "news": ["news_id", "publisher_id", "title", "language", "category", "published_at", "article_text_len"],
}

masked_samples = {}
for table, cols in sample_columns.items():
    df = data[table]
    cols = [c for c in cols if c in df.columns]
    # Prefer rows where the key analysis labels are nonblank, so the sample is pedagogically useful.
    sample = df[cols].copy()
    if table == "clicks" and "news_matched" in sample.columns:
        sample = sample.loc[sample["news_matched"].astype(str).str.lower().isin(["true", "1"])]
    if table == "recommendation_items" and "news_matched" in sample.columns:
        sample = sample.loc[sample["news_matched"].astype(str).str.lower().isin(["true", "1"])]
    sample = sample.head(3).copy()
    for col in sample.columns:
        sample[col] = sample[col].map(lambda v, col=col: mask_value(col, v))
    masked_samples[table] = sample
    display(Markdown(f"### `{table}` masked sample"))
    display(sample)
    sample.to_csv(SCRATCH_TABLES / f"sample_{table}_masked.csv", index=False)


## Join map: how raw traces become comparable PEC layers

PEC 분석은 “사용자”, “기사”, “날짜”라는 세 축을 반복해서 연결한다.

- 사용자 축: `profiles.user_id` ↔ `clicks.user_id` ↔ `recommendation_items.user_id`
- 기사 축: `clicks.news_id` ↔ `recommendation_items.news_id` ↔ `news.news_id`
- 날짜 축: `clicks.event_date_iso` ↔ `recommendation_items.recommendation_date`

이 세 축이 모두 있어야 RQ2의 same-day traceability 같은 분석이 가능하다. 반대로 어떤 축이 빠지면 그것은 데이터 오류라기보다 observability boundary가 된다.

In [ ]:
join_map = pd.DataFrame([
    {
        "join step": "clicks + news metadata",
        "left table": "click_events_enriched.csv",
        "right table": "news_master.csv",
        "join keys": "news_id",
        "purpose": "clicked article에 category/publisher/published_at 같은 공통 label을 붙인다.",
        "output meaning": "Consumption C를 category/source distribution으로 바꿀 수 있다.",
    },
    {
        "join step": "recommendation items + news metadata",
        "left table": "recommendation_items_enriched.csv",
        "right table": "news_master.csv",
        "join keys": "news_id",
        "purpose": "추천된 article item에도 동일한 category/publisher label을 붙인다.",
        "output meaning": "Logged exposure E를 click C와 같은 label space에서 비교할 수 있다.",
    },
    {
        "join step": "profiles + clicked consumption",
        "left table": "user_profiles_clean.csv",
        "right table": "click_events_enriched.csv",
        "join keys": "user_id",
        "purpose": "사용자가 말하거나 유지한 preference와 실제 clicked category/source를 비교한다.",
        "output meaning": "RQ1 P-C alignment: top-category match, Jaccard, weighted cosine.",
    },
    {
        "join step": "clicks + same-day recommendation exposure",
        "left table": "click_events_enriched.csv",
        "right table": "recommendation_items_enriched.csv",
        "join keys": "user_id + date + news_id",
        "purpose": "클릭된 기사가 같은 사용자의 같은 날짜 logged recommendation list에 있었는지 확인한다.",
        "output meaning": "RQ2 E-C traceability: same-day hit, top-k, candidate-pool baseline.",
    },
    {
        "join step": "click distribution + recommendation distribution",
        "left table": "click_events_enriched.csv",
        "right table": "recommendation_items_enriched.csv",
        "join keys": "user_id, then aggregate category/publisher distributions",
        "purpose": "같은 사용자에 대해 exposure distribution과 click distribution을 만든다.",
        "output meaning": "RQ3 diversity/concentration shift: entropy, HHI, top-share, JS.",
    },
    {
        "join step": "RQ summaries + profile/exposure/behavior features",
        "left table": "RQ1/RQ2/RQ3 user-level summaries",
        "right table": "profiles, exposure summaries, click-surface summaries",
        "join keys": "user_id",
        "purpose": "preference-consumption divergence가 observable features로 구조화되는지 본다.",
        "output meaning": "RQ4 diagnostic model feature matrix.",
    },
])
display(join_map)


## Analysis-table flow: which intermediate table supports each RQ?

논문 결과는 한 번에 계산되는 것이 아니라, 아래처럼 중간 산출물로 나뉜다. 이걸 이해하면 “어느 숫자가 어디서 왔는지”를 따라가기 쉬워진다.

In [ ]:
analysis_table_flow = pd.DataFrame([
    {
        "RQ": "RQ1 P-C alignment",
        "main constructed unit": "one row per user",
        "inputs": "profiles + matched click rows + news labels",
        "transformation": "stated category/source sets and weighted profile vectors are compared with clicked category/source sets/distributions",
        "typical output table": "rq1_preference_consumption_gap_by_user.csv; preference_weighted_features.csv",
        "paper quantities": "top-category match, category Jaccard, weighted category/source cosine",
    },
    {
        "RQ": "RQ2 E-C traceability",
        "main constructed unit": "one row per clicked article event, then user/surface summaries",
        "inputs": "matched click rows + recommendation item rows + dates + ranks",
        "transformation": "for each click, check whether same user's same-day recommendation set contained the clicked news_id; compare to candidate-pool baselines",
        "typical output table": "rq2_click_exposure_rows.csv; optimization_topk_exposure.csv; surface traceability summaries",
        "paper quantities": "same-day hit rate, six-month recommendation-history overlap, top-k lift/gap, surface composition",
    },
    {
        "RQ": "RQ3 E-C diversity shift",
        "main constructed unit": "one row per user distribution summary",
        "inputs": "user-level recommendation item lists + user-level click lists",
        "transformation": "build exposure and click distributions over category/source labels; compare raw and count-matched distributions",
        "typical output table": "rq3_diversity_by_user.csv; count-matched diversity tables",
        "paper quantities": "entropy gap, HHI gap, top-share gap, JS divergence",
    },
    {
        "RQ": "RQ4 structured divergence",
        "main constructed unit": "one row per user feature vector",
        "inputs": "RQ1 divergence label + profile features + exposure features + behavior/surface features + weighted-profile features",
        "transformation": "train diagnostic models to test whether high divergence is structured under bounded validation checks",
        "typical output table": "rq4_divergence_model_features.csv; model CV/stress tables",
        "paper quantities": "same-window AUC, dense-history AUC, future-window AUC, label-proximity sensitivity",
    },
])
display(analysis_table_flow)


## How to read the rest of this EDA notebook

After the data guide, the remaining cells check whether the inputs are healthy enough to support the paper:

1. **Validation summary:** whether previous pipeline checks passed.
2. **Date ranges:** whether rows cover the intended study window.
3. **Missingness:** whether key join/label fields are blank.
4. **Core counts:** how many profiles, matched clicks, matched recommendation items, and users exist.
5. **Monthly volume:** whether logs and metadata have plausible temporal coverage.

이 순서를 거치고 나서야 01번 노트북에서 실제 RQ1/RQ2 조인을 재구성한다.

In [ ]:
# AUTO-SAVE REPRO_DEEP SCRATCH OUTPUTS
# This cell writes notebook-generated rerun outputs only to tmp/repro_deep/...
# It does not write to reports/tables, reports/figures, or the Overleaf submission folder.
_save_names = ['shapes', 'dataset_guide', 'inventory', 'column_dictionary', 'join_map', 'analysis_table_flow', 'date_checks', 'core_counts', 'monthly']
_saved_paths = []
for _name in _save_names:
    if _name not in globals():
        continue
    _obj = globals()[_name]
    _path = SCRATCH_TABLES / f"{_name}.csv"
    try:
        if isinstance(_obj, pd.DataFrame):
            _obj.to_csv(_path, index=False)
        elif isinstance(_obj, pd.Series):
            _obj.to_frame().to_csv(_path)
        elif isinstance(_obj, list):
            pd.DataFrame(_obj).to_csv(_path, index=False)
        elif isinstance(_obj, dict):
            pd.DataFrame([_obj]).to_csv(_path, index=False)
        else:
            continue
        _saved_paths.append(_path)
    except Exception as exc:
        print(f"Could not save {_name}: {exc}")

try:
    import matplotlib.pyplot as _plt
    for _idx, _num in enumerate(_plt.get_fignums(), 1):
        _fig = _plt.figure(_num)
        _fig_path = SCRATCH_FIGURES / f"{NOTEBOOK_STEM}_figure_{_idx:02d}.png"
        _fig.savefig(_fig_path, dpi=160, bbox_inches="tight")
        _saved_paths.append(_fig_path)
except Exception as exc:
    print(f"Could not save open matplotlib figures: {exc}")

print("Saved scratch artifacts:")
for _path in _saved_paths:
    print(" -", _path)
